# 第04章：Block 操作 — Triton 的核心编程范式

## 前置知识
- 第01-03章

## 学习目标
- 深入理解 `tl.arange` 和块级操作
- 掌握 Triton 的广播（broadcasting）规则
- 学会构建 2D 索引模式
- 理解块级编程的思维方式

In [ ]:
import torch
import triton
import triton.language as tl

## 4.1 tl.arange — Triton 的核心原语

`tl.arange(start, end)` 是 Triton 中最基本的操作，类似 `range()`，但生成的是**编译时固定大小**的张量：

```python
tl.arange(0, 4)  →  [0, 1, 2, 3]   # 形状: (4,)
tl.arange(0, 8)  →  [0, 1, 2, 3, 4, 5, 6, 7]  # 形状: (8,)
```

**重要限制**: `end - start` 必须是 **2 的幂**！
```python
tl.arange(0, 4)   ✅  (4 = 2²)
tl.arange(0, 8)   ✅  (8 = 2³)
tl.arange(0, 5)   ❌  (5 不是 2 的幂)
```

### 为什么必须是 2 的幂？
因为 Triton 编译器会将块操作映射到 GPU warp（32个线程），2 的幂大小能最高效地利用硬件。

## 4.2 广播（Broadcasting）

Triton 的广播规则与 NumPy/PyTorch 类似：

```
1D + 标量:   [a,b,c,d] + s     → [a+s, b+s, c+s, d+s]
1D + 1D:    [a,b,c,d] + [e,f,g,h] → [a+e, b+f, c+g, d+h]

2D 广播 (创建 2D 索引的关键):

列向量 (4,1):     行向量 (1,4):        结果 (4,4):
┌───┐             ┌───┬───┬───┬───┐    ┌────┬────┬────┬────┐
│ 0 │             │ 0 │ 1 │ 2 │ 3 │    │0+0 │0+1 │0+2 │0+3 │
│ 1 │      +      └───┴───┴───┴───┘ =  │1+0 │1+1 │1+2 │1+3 │
│ 2 │                                   │2+0 │2+1 │2+2 │2+3 │
│ 3 │                                   │3+0 │3+1 │3+2 │3+3 │
└───┘                                   └────┴────┴────┴────┘

代码:
rows = tl.arange(0, 4)[:, None]  # (4, 1) 列向量
cols = tl.arange(0, 4)[None, :]  # (1, 4) 行向量
matrix = rows + cols              # (4, 4) 广播
```

### `[:, None]` 和 `[None, :]` 的含义
- `[:, None]` → 在第1维插入新轴 → (N,) 变成 (N, 1) → 列向量
- `[None, :]` → 在第0维插入新轴 → (N,) 变成 (1, N) → 行向量

In [ ]:
@triton.jit
def broadcast_demo_kernel(out_ptr, N: tl.constexpr):
    """
    演示广播：生成一个 N×N 的矩阵，其中 out[i,j] = i * N + j
    这就是 row-major 线性索引!
    """
    rows = tl.arange(0, N)  # [0, 1, ..., N-1]
    cols = tl.arange(0, N)  # [0, 1, ..., N-1]
    
    # 广播: (N, 1) * scalar + (1, N) → (N, N)
    linear_idx = rows[:, None] * N + cols[None, :]
    
    # 存储到连续内存
    offsets = rows[:, None] * N + cols[None, :]
    tl.store(out_ptr + offsets, linear_idx.to(tl.float32))

N = 4
out = torch.empty(N, N, device='cuda')
broadcast_demo_kernel[(1,)](out, N=N)
print("广播生成的线性索引矩阵:")
print(out.int())

## 4.3 常用的 2D 索引模式

在 GEMM 和 Attention 中，最常用的 2D 索引模式：

In [ ]:
@triton.jit
def index_patterns_kernel(
    out_add_ptr, out_mul_ptr, out_mask_ptr,
    N: tl.constexpr,
):
    """
    三种常用 2D 模式：
    1. 加法索引 (内存偏移计算)
    2. 乘法索引 (外积)
    3. 比较 mask (上三角/下三角)
    """
    rows = tl.arange(0, N)
    cols = tl.arange(0, N)
    offsets = rows[:, None] * N + cols[None, :]
    
    # 模式1: 加法 (用于计算线性内存地址)
    # out[i,j] = i + j
    add_pattern = rows[:, None] + cols[None, :]
    tl.store(out_add_ptr + offsets, add_pattern.to(tl.float32))
    
    # 模式2: 乘法 (外积, 用于 rank-1 更新)
    # out[i,j] = i * j
    mul_pattern = rows[:, None] * cols[None, :]
    tl.store(out_mul_ptr + offsets, mul_pattern.to(tl.float32))
    
    # 模式3: 比较 mask (用于 causal attention)
    # out[i,j] = 1 if i >= j else 0  (下三角)
    mask_pattern = (rows[:, None] >= cols[None, :]).to(tl.float32)
    tl.store(out_mask_ptr + offsets, mask_pattern)

N = 4
out_add = torch.empty(N, N, device='cuda')
out_mul = torch.empty(N, N, device='cuda')
out_mask = torch.empty(N, N, device='cuda')

index_patterns_kernel[(1,)](out_add, out_mul, out_mask, N=N)

print("模式1 - 加法索引 (i+j):")
print(out_add.int())
print("\n模式2 - 外积 (i*j):")
print(out_mul.int())
print("\n模式3 - 下三角 mask (i>=j):")
print(out_mask.int())
print("↑ 这个 mask 就是 Causal Attention 中的因果 mask！")

## 4.4 块级元素操作

Triton 的所有操作都是**块级**的，意味着对整个块的所有元素同时执行：

```python
# 假设 a = [1.0, 2.0, 3.0, 4.0]
a + 1.0    # → [2.0, 3.0, 4.0, 5.0]  (标量广播)
a * 2.0    # → [2.0, 4.0, 6.0, 8.0]
a + b      # → 逐元素加 (a 和 b 形状必须相同或可广播)
a > 2.0    # → [False, False, True, True]  (逐元素比较)
```

支持的操作：
- 算术: `+`, `-`, `*`, `/`, `//`, `%`, `**`
- 比较: `>`, `<`, `>=`, `<=`, `==`, `!=`
- 逻辑: `&`, `|`, `^`, `~`
- 位运算: `<<`, `>>`

In [ ]:
@triton.jit
def elementwise_ops_kernel(
    a_ptr, out_relu_ptr, out_clamp_ptr, out_norm_ptr,
    n_elements, BLOCK_SIZE: tl.constexpr,
):
    """
    常用的块级元素操作示例:
    1. ReLU: max(x, 0)
    2. Clamp: min(max(x, -1), 1)
    3. 归一化: x / max(|x|)
    """
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    
    a = tl.load(a_ptr + offsets, mask=mask)
    
    # ReLU: 使用 tl.where 或直接用 tl.maximum
    relu = tl.where(a > 0, a, 0.0)  # 等价于 tl.maximum(a, 0.0)
    tl.store(out_relu_ptr + offsets, relu, mask=mask)
    
    # Clamp: 限制到 [-1, 1]
    clamped = tl.minimum(tl.maximum(a, -1.0), 1.0)
    tl.store(out_clamp_ptr + offsets, clamped, mask=mask)
    
    # 归一化: 除以块内最大绝对值
    abs_a = tl.abs(a)
    max_val = tl.max(abs_a, axis=0)
    normalized = a / (max_val + 1e-6)  # 加 epsilon 防止除零
    tl.store(out_norm_ptr + offsets, normalized, mask=mask)

n = 16
a = torch.randn(n, device='cuda') * 3
out_relu = torch.empty_like(a)
out_clamp = torch.empty_like(a)
out_norm = torch.empty_like(a)

elementwise_ops_kernel[(1,)](a, out_relu, out_clamp, out_norm, n, BLOCK_SIZE=16)

print(f"输入:       {a[:8].tolist()}")
print(f"ReLU:      {out_relu[:8].tolist()}")
print(f"Clamp:     {out_clamp[:8].tolist()}")
print(f"Normalize: {out_norm[:8].tolist()}")

## 4.5 tl.zeros 和 tl.full — 创建常量块

```python
# 全零块
zeros = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

# 全1块
ones = tl.full((BLOCK_SIZE,), value=1.0, dtype=tl.float32)

# 负无穷块 (softmax 中用于初始化最大值)
neg_inf = tl.full((BLOCK_SIZE,), value=float('-inf'), dtype=tl.float32)
```

这些在累加器初始化和 attention mask 中非常常用。

In [ ]:
@triton.jit
def init_patterns_kernel(out_ptr, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    """演示各种初始化模式"""
    # 全零 (GEMM 累加器初始化)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    
    # 单位矩阵 (通过条件赋值)
    rows = tl.arange(0, BLOCK_M)
    cols = tl.arange(0, BLOCK_N)
    identity = tl.where(
        rows[:, None] == cols[None, :],  # 对角线条件
        1.0,
        0.0,
    )
    
    # 存储单位矩阵
    offsets = rows[:, None] * BLOCK_N + cols[None, :]
    tl.store(out_ptr + offsets, identity)

N = 4
out = torch.empty(N, N, device='cuda')
init_patterns_kernel[(1,)](out, BLOCK_M=N, BLOCK_N=N)
print("生成的单位矩阵:")
print(out)

## 总结

| 操作 | API | 说明 |
|------|-----|------|
| 整数序列 | `tl.arange(start, end)` | end-start 必须是 2 的幂 |
| 列向量 | `arr[:, None]` | 形状 (N,) → (N, 1) |
| 行向量 | `arr[None, :]` | 形状 (N,) → (1, N) |
| 全零 | `tl.zeros(shape, dtype)` | 初始化累加器 |
| 全值 | `tl.full(shape, val, dtype)` | 初始化常量 |
| 条件选择 | `tl.where(cond, x, y)` | 等价于 np.where |
| 最大/最小 | `tl.maximum(a, b)` / `tl.minimum(a, b)` | 逐元素 |

### 核心思维转变

```
CUDA 思维:    "对每个元素做 X"       → for 循环，逐个处理
Triton 思维:  "对整个数据块做 X"     → 向量化操作，一次处理一块
```

## 练习

1. 用广播生成一个 **乘法表** (8×8)
2. 实现 **Leaky ReLU**: `f(x) = x if x > 0 else 0.01 * x`
3. 实现 **上三角 mask**: `mask[i,j] = 1 if i <= j else 0`
4. 用块操作实现 **矩阵对角线缩放**: 将矩阵的对角线元素乘以 2